# 1109 47일차

## 1. HalvingGridSearchCV

적은 데이터로 조합을 먼저 걸러 내고 남은 조합에만 데이터를 늘려 가며 탐색하는 것 (`m10_HalvingGridSearchCV_00`)

`GridSearchCV`는 조합마다 train 전체를 쓰기 때문에 후보를 늘리는 만큼 시간이 늘어남. 나쁜 조합은 데이터가 적어도 걸러지니 초반 iter에 샘플을 적게 줘서 시간을 줄임

1. `from sklearn.experimental import enable_halving_search_cv` : 아직 실험 기능이라 이 줄을 불러 둬야 `sklearn.model_selection`에서 꺼낼 수 있음
2. `HalvingGridSearchCV(xgb, parameters, cv=kfold, verbose=1, refit=True)` : `GridSearchCV` 자리에 그대로 바꿔 씀
3. `factor=2` : 다음 iter에 남길 조합 비율과 늘릴 샘플 배수. 조합은 1/2로 줄고 샘플은 2배가 됨. 기본값은 3
4. `min_resources=100` : 첫 iter에서 조합마다 쓸 샘플 수
5. `max_resources=1437` : 마지막 iter에서 쓸 수 있는 샘플 수의 상한. 기본값은 train 전체

- iter마다 점수가 낮은 조합을 버리고 남은 조합에 샘플을 더 줌
- `min_resources` 기본값 `'exhaust'`는 마지막 iter가 `max_resources`를 최대한 쓰도록 거꾸로 계산함
- `'exhaust'`의 하한은 `n_splits * 2 * 클래스 수`. 5-fold에 digits 클래스가 10개면 100
- **`× 2`는 fold마다 클래스당 샘플이 몇 개는 들어가게 두려고 붙은 값이고, 조합을 줄이는 `factor`와는 다름**
- `n_required_iterations`는 조합이 1개로 줄 때까지 필요한 횟수, `n_possible_iterations`는 샘플 수로 가능한 횟수. 둘 중 작은 쪽이 `n_iterations`
- 조합이 1개로 줄기 전에 멈추면 마지막 iter에 남은 조합 중에서 `best_params_`를 고름
- `factor`를 줄이면 한 번에 버리는 조합이 적어져서 iter 수와 시간이 늘어남

#### cf) 적은 데이터

- 초반 iter는 샘플이 적어서 조합의 점수가 흔들림. 적은 샘플에 맞춰진 조합이 살아남고 좋은 조합이 먼저 걸러질 수 있음
- `min_resources`를 올려서 보완하지만, 전체 데이터가 적으면 `max_resources`에 금방 닿아서 iter를 늘릴 여유가 없음
- **데이터가 적으면 Halving으로 줄일 시간도 적고 걸러내는 기준만 흔들리므로 전체를 쓰는 탐색을 씀**